# [초급 프로젝트] 4팀_김명환 - 데이타 전처리

---
---

# 환경설정
    - 라이브러리 설치 및 로딩
    - 사용자 함수 사용

In [1]:
# 기본 라이브러리

# --- Scikit-learn: 데이터 전처리, 모델, 평가 ---
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import (
    fetch_california_housing, load_iris, make_moons, make_circles,
    load_breast_cancer, load_wine
)
from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.metrics import average_precision_score

# --- 기타 라이브러리 ---
import cv2
from PIL import Image
from PIL import ImageFilter
from PIL import ImageDraw
import albumentations as A
import IPython.display
#from tqdm import tqdm
from tqdm.notebook import tqdm

# --- PyTorch: 딥러닝 관련 ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torch.utils.data import Subset
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as TF
from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset
from collections import OrderedDict
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask
import torchvision.models as models
from collections import defaultdict
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances

# --- 기타 ---
import re
import os
import sys
import copy
import json
import math
import random
import yaml
import shutil
import pandas as pd
import numpy as np
import seaborn as sns
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from datetime import datetime
from datetime import timezone, timedelta
import pytz
__kst = pytz.timezone('Asia/Seoul')

# GPU 설정
__device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
__device_cpu = torch.device('cpu')

  # 재현 가능한 결과를 위해
np.random.seed(42)
torch.manual_seed(42)
if __device == 'cuda':
    torch.cuda.manual_seed_all(42)

print(f"라이브러리 로드 완료 사용장치:{__device}")

라이브러리 로드 완료 사용장치:cpu


In [2]:
from urllib.request import urlretrieve; urlretrieve("https://raw.githubusercontent.com/c0z0c/codeit_ai_health_eat/refs/heads/alpha/src/python_modules/utils/health_ea_utils.py", 
                                                    "health_ea_utils.py")
import importlib
import health_ea_utils as heu
importlib.reload(heu)
import helper_c0z0c_dev as helper
importlib.reload(helper)
from health_ea_utils import *
import helper_c0z0c_dev as helper


🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.1.3 (v2.x+): 호환성 모드 적용됨
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = d:\GoogleDrive\codeit_ai_health_eat\scripts\김명환
🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.1.3 (v2.x+): 호환성 모드 적용됨
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = d:\GoogleDrive\codeit_ai_health_eat\scripts\김명환
🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.1.3 (v2.x+): 호환성 모드 적용됨
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = d:\GoogleDrive\codeit_ai_health_eat\scripts\김명환
🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.1.3 (v2.x+): 호환성 모드 적용됨
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = d:\GoogleDrive\codeit_ai_health_eat\scripts\김명환


# 프로그래밍

## 1. 학습용 데이타 다운로드 및 압축 풀기

In [3]:
# 코드잇의 데이타 파일을 다운로드 받고 파일 목록을 DataFrame으로 반환
def df_filename_list(paths):
    """
    paths 하위의 모든 .json 파일에 대해
    - 파일명(확장자 없는)
    - json 파일 경로
    - png 파일 경로 (동일 경로, 동일 파일명, 확장자만 .png)
    를 DataFrame으로 반환
    """
    import os
    import pandas as pd

    records = []
    if isinstance(paths, str):
        paths = [paths]
    for root in paths:
        for dirpath, _, filenames in os.walk(root):  # 여기서 root로 변경
            for fname in filenames:
                drug_info={
                    'filename': None,
                    'ext': None,
                    'file_name': None,
                    'path': None,
                    'label': None,
                    'drug0': None,
                    'drug1': None,
                    'drug2': None,
                    'drug3': None,
                }

                filename, ext = os.path.splitext(fname)
                
                ext = ext.lower().replace('.', '')  # 확장자에서 . 제거
                if ext not in ['png', 'jpg', 'jpeg', 'json']:
                    continue  # png, jpg, jpeg만 조회

                drug_info.update({
                    'filename': filename,
                    'file_name': fname,
                    'ext': ext,
                    'path': os.path.join(dirpath, fname),
                })
                if filename.startswith('K-'):
                    # 예시: K-001900-010224-016551-031705_0_2_0_2_70_000_200
                    parts = filename.split('_')[0].split('-')
                    if len(parts) >= 5:
                        drug_info.update({
                            'label': f'{filename.split("_")[0]}',
                            'drug0': f'K-{parts[1]}',
                            'drug1': f'K-{parts[2]}',
                            'drug2': f'K-{parts[3]}',
                            'drug3': f'K-{parts[4]}',
                        })
                records.append(drug_info)
    return pd.DataFrame(records)


In [4]:
image_dir = r"D:\dataset\166.약품식별 인공지능 개발을 위한 경구약제 이미지 데이터\01.데이터\1.Training\원천데이터\경구약제조합 5000종"
json_dir = r"D:\dataset\166.약품식별 인공지능 개발을 위한 경구약제 이미지 데이터\01.데이터\1.Training\라벨링데이터\경구약제조합 5000종"
data_dir = r"D:\dataset\kaggle_code_it_data\ai04-level1-project.zip.unzip_train"
#df_files = df_filename_list([data_dir])
df_files = df_filename_list([image_dir, json_dir, data_dir])
print("df_files",len(df_files))
print(df_files['ext'].value_counts())



df_files 66085
ext
json    50599
png     15486
Name: count, dtype: int64


In [5]:
# 일부 목록을 검토
df_files

,filename,ext,file_name,path,label,drug0,drug1,drug2,drug3
0,K-000250-000573-002483-006192_0_2_0_2_70_000_200,png,K-000250-000573-002483-006192_0_2_0_2_70_000_2...,D:\dataset\166.약품식별 인공지능 개발을 위한 경구약제 이미지 데이터\0...,K-000250-000573-002483-006192,K-000250,K-000573,K-002483,K-006192
1,K-000250-000573-002483-006192_0_2_0_2_75_000_200,png,K-000250-000573-002483-006192_0_2_0_2_75_000_2...,D:\dataset\166.약품식별 인공지능 개발을 위한 경구약제 이미지 데이터\0...,K-000250-000573-002483-006192,K-000250,K-000573,K-002483,K-006192
2,K-000250-000573-002483-006192_0_2_0_2_90_000_200,png,K-000250-000573-002483-006192_0_2_0_2_90_000_2...,D:\dataset\166.약품식별 인공지능 개발을 위한 경구약제 이미지 데이터\0...,K-000250-000573-002483-006192,K-000250,K-000573,K-002483,K-006192
3,K-000250-000573-002483-006192_index,png,K-000250-000573-002483-006192_index.png,D:\dataset\166.약품식별 인공지능 개발을 위한 경구약제 이미지 데이터\0...,K-000250-000573-002483-006192,K-000250,K-000573,K-002483,K-006192
4,K-000250-000573-002483-012778_0_2_0_2_70_000_200,png,K-000250-000573-002483-012778_0_2_0_2_70_000_2...,D:\dataset\166.약품식별 인공지능 개발을 위한 경구약제 이미지 데이터\0...,K-000250-000573-002483-012778,K-000250,K-000573,K-002483,K-012778
...,...,...,...,...,...,...,...,...,...
66080,K-003544-012247-016548-021026_0_2_0_2_75_000_200,png,K-003544-012247-016548-021026_0_2_0_2_75_000_2...,D:\dataset\kaggle_code_it_data\ai04-level1-pro...,K-003544-012247-016548-021026,K-003544,K-012247,K-016548,K-021026
66081,K-003544-012247-016548-021026_0_2_0_2_90_000_200,png,K-003544-012247-016548-021026_0_2_0_2_90_000_2...,D:\dataset\kaggle_code_it_data\ai04-level1-pro...,K-003544-012247-016548-021026,K-003544,K-012247,K-016548,K-021026
66082,K-003544-012247-016548-027926_0_2_0_2_70_000_200,png,K-003544-012247-016548-027926_0_2_0_2_70_000_2...,D:\dataset\kaggle_code_it_data\ai04-level1-pro...,K-003544-012247-016548-027926,K-003544,K-012247,K-016548,K-027926
66083,K-003544-012247-016548-027926_0_2_0_2_75_000_200,png,K-003544-012247-016548-027926_0_2_0_2_75_000_2...,D:\dataset\kaggle_code_it_data\ai04-level1-pro...,K-003544-012247-016548-027926,K-003544,K-012247,K-016548,K-027926


## 2. Yolo DataFrame

In [6]:
# json 파일을 DataFrame로 로드
df = df_files.copy()
def json_to_df(json_path):
    """
    json 파일에서 images, annotations, categories 정보를 DataFrame으로 반환
    """
    with open(json_path, encoding='utf-8') as f:
        data = json.load(f)

    df_images = pd.DataFrame(data.get('images', []))
    df_annotations = pd.DataFrame(data.get('annotations', []))
    df_categories = pd.DataFrame(data.get('categories', []))

    return df_images, df_annotations, df_categories

def bbox_to_yolo(bbox, img_width, img_height):
    """
    COCO bbox를 YOLO 형식(x_center, y_center, w, h)로 변환
    """
    # bbox: [x, y, w, h] (COCO)
    if not bbox or len(bbox) < 4:
        return None  # 오류시 None 반환

    x, y, w, h = bbox[:4]
    x_center = (x + w / 2) / img_width
    y_center = (y + h / 2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    return x_center, y_center, w_norm, h_norm

def collect_json_info(df):
    """
    파일 목록 DataFrame(df)에서 json을 읽어 이미지/라벨/약품 정보를 통합
    - 오류 파일은 error_files에 기록
    - bbox, yolo 좌표, class_id 등 추가
    """
    import pandas as pd

    records = []
    drug_info = {}
    error_files = []  # 오류 파일 목록

    # Test 분류
    df_test = df[(df['ext'] == 'png') & (df['path'].str.contains('test', case=False))].copy()
    print("df_test", len(df_test))
    pbar = tqdm(df_test.iterrows(), total=len(df_test), mininterval=3, desc="Processing Test files")
    for idx, row in pbar:
        field = {
                **row,
                'imgfile': row['path'],
                'Train' : False,
                'Test' : True,
                'drug_N': None,
                'width': 0,
                'height': 0,
                'bbox_x': 0,
                'bbox_y': 0,
                'bbox_w': 0,
                'bbox_h': 0,
                'yolo_x': 0.0,
                'yolo_y': 0.0,
                'yolo_w': 0.0,
                'yolo_h': 0.0
            }
        filename_without_ext = os.path.splitext(os.path.basename(row['path']))[0]
        field['filename'] = filename_without_ext
        records.append(field)

    # Train 분류
    df_json = df[df['ext'] == 'json'].copy()
    df_png = df[df['ext'] == 'png'].copy()
    print("png, json", len(df_png), len(df_json))
    pbar = tqdm(df_json.iterrows(), total=len(df_json), mininterval=3, desc="Processing JSON files")
    for idx, row in pbar:
        field = {
                **row,
                'imgfile': None,
                'Train' : True,
                'Test' : False,
                'drug_N': None,
                'width': 0,
                'height': 0,
                'bbox_x': 0,
                'bbox_y': 0,
                'bbox_w': 0,
                'bbox_h': 0,
                'yolo_x': 0.0,
                'yolo_y': 0.0,
                'yolo_w': 0.0,
                'yolo_h': 0.0
            }
        filename = row['filename']
        json_path = row['path']

        try:
            df_images, df_annotations, df_categories = json_to_df(json_path)
        except Exception as e:
            print(f"JSON 파싱 오류 - 파일: {filename}, 오류: {e}")
            error_files.append(filename)
            continue

        if df_images.empty or df_annotations.empty:
            print(f"데이터 부족 - 파일: {filename} (images: {len(df_images)}, annotations: {len(df_annotations)})")
            error_files.append(filename)
            continue

        img_row = df_images.iloc[0]
        ann_row = df_annotations.iloc[0]
        cat_row = df_categories.iloc[0] if not df_categories.empty else {}

        png_match = df_png[df_png['file_name'] == img_row.get('file_name', None)]
        if png_match.empty:
            pbar.set_postfix_str(f"이미지 파일 없음 - 파일: {filename}")
            error_files.append(filename)
            continue

        field['imgfile'] = png_match['path'].values[0]  # path가 여러 개면 첫 번째 값 사용

        filename_without_ext = os.path.splitext(os.path.basename(png_match['path'].iloc[0]))[0]
        field['filename'] = filename_without_ext

        if pd.isna(field['imgfile']) | (field['imgfile'] is None):
            pbar.set_postfix_str(f"이미지 파일명 누락 - 파일: {filename}")
            error_files.append(filename)
            continue

        # bbox 검증
        bbox = ann_row.get('bbox', [])
        if not bbox or len(bbox) < 4:
            print(f"bbox 오류 - 파일: {filename}, bbox: {bbox}")
            error_files.append(filename)
            continue

        # YOLO bbox 계산
        yolo_result = bbox_to_yolo(bbox, img_row['width'], img_row['height'])
        if yolo_result is None:
            print(f"YOLO 변환 오류 - 파일: {filename}")
            error_files.append(filename)
            continue

        x_center, y_center, w_norm, h_norm = yolo_result

        # 기존 df row에 정보 추가
        field.update({
            'drug_N': img_row.get('drug_N'),
            'category_id': ann_row.get('category_id'),
            'width': img_row.get('width'),
            'height': img_row.get('height'),
            'bbox_x': bbox[0],
            'bbox_y': bbox[1],
            'bbox_w': bbox[2],
            'bbox_h': bbox[3],
            'yolo_x': x_center,
            'yolo_y': y_center,
            'yolo_w': w_norm,
            'yolo_h': h_norm
        })
        records.append(field)

        # 약 정보 dict (중복 제거)
        drug_N = img_row.get('drug_N')
        if drug_N and drug_N not in drug_info:
            drug_info[drug_N] = {
                'drug_N': drug_N,
                'category_id': ann_row.get('category_id'),
                'drug_S': img_row.get('drug_S'),
                'dl_name': img_row.get('dl_name'),
                'dl_name_en': img_row.get('dl_name_en'),
                'img_key': img_row.get('img_key'),
                'dl_material': img_row.get('dl_material'),
                'dl_material_en': img_row.get('dl_material_en'),
                'dl_custom_shape': img_row.get('dl_custom_shape'),
                'dl_company': img_row.get('dl_company'),
                'dl_company_en': img_row.get('dl_company_en'),
                'di_class_no': img_row.get('di_class_no'),
                'di_etc_otc_code': img_row.get('di_etc_otc_code'),
                'di_edi_code': img_row.get('di_edi_code'),
                'chart': img_row.get('chart'),
                'drug_shape': img_row.get('drug_shape'),
                'form_code_name': img_row.get('form_code_name'),
                'supercategory': cat_row.get('supercategory', ''),
                'name': cat_row.get('name', '')
            }
        if idx % 100 == 0:
            pbar.set_postfix_str(filename)

    print(f"\n=== 처리 결과 ===")
    print(f"전체 파일: {len(df)}")
    print(f"성공 처리: {len(records)}")
    print(f"오류 파일: {len(error_files)}")
    if error_files:
        print(f"오류 파일 목록 (처음 10개): {error_files[:10]}")

    df_new = pd.DataFrame(records)
    df_drug = pd.DataFrame(list(drug_info.values()))

    train_df = df_new[df_new['Train'] == True]
    drug_classes = {drug_N: idx+1 for idx, drug_N in enumerate(sorted(train_df['drug_N'].unique()))}

    df_new['class_id'] = train_df['drug_N'].map(drug_classes).fillna(0).astype(int)
    df_drug['class_id'] = df_drug['drug_N'].map(drug_classes).fillna(0).astype(int)

    # df_drug = pd.DataFrame(list(drug_info.values()))
    # drug_classes = {drug_N: idx+1 for idx, drug_N in enumerate(sorted(df_new['drug_N'].unique()))}
    # df_new['class_id'] = df_new['drug_N'].map(drug_classes).fillna(0).astype(int)
    # df_drug['class_id'] = df_drug['drug_N'].map(drug_classes).fillna(0).astype(int)

    return df_new, df_drug, drug_classes

# os.path.join(kaggle_unzip_path, 'train_images')
def create_yolo_dataset(df_files, ignore=True):
    """
    파일 목록에서 YOLO 학습용 DataFrame, 약품 정보 DataFrame 생성 및 저장
    """
    df = helper.pd_checkout("df_codeit04_new", commit_dir=drive_root())
    df_drug = helper.pd_checkout("df_codeit04_drug", commit_dir=drive_root())
    if df.empty or df_drug.empty or ignore:
        from datetime import datetime
        print("JSON 정보 수집 중...")
        # 실행
        df, df_drug, _ = collect_json_info(df_files)
        print('df_new shape:', df.shape)
        print('df_drug shape:', df_drug.shape)

        helper.pd_commit(df, "df_codeit04_new", commit_dir=drive_root())
        helper.pd_commit(df_drug, "df_codeit04_drug", commit_dir=drive_root())

        print("df_codeit04_new, df_codeit04_drug 저장")
    else:
        print("이미 df_codeit04_new, df_codeit04_drug가 존재함")

    df.sort_values(by='filename',inplace=True)
    df_drug.sort_values(by='drug_N',inplace=True)
    return df, df_drug



In [7]:
# drug_N에서 - 뒤 숫자에서 -1을 해서 category_id로 지정
def drugN_to_category_id(drug_N):
    try:
        # drug_N이 'K-000250' 형태라면, '-' 뒤 숫자 추출
        num = int(drug_N.split('-')[1])
        return num - 1
    except Exception:
        return None


In [8]:
raise ValueError("stop")

ValueError: stop

In [ ]:

# JSON 정보를 수집하여 DataFrame 생성
df_train_test, df_drug = create_yolo_dataset(df_files = df_files, ignore=True)

df_train_test['category_id'] = df_train_test['drug_N'].apply(drugN_to_category_id)
df_drug['category_id'] = df_drug['drug_N'].apply(drugN_to_category_id)

drug_classes = dict(zip(df_drug['drug_N'], df_drug['class_id']))
drug_classes_idx = dict(zip(df_drug['class_id'], df_drug['drug_N']))

In [ ]:
df_train_test.to_pickle(r"D:\dataset\kaggle_code_it_data_경구약제조합_5000종_250922_0952.pkl")
df_drug.to_pickle(r"D:\dataset\kaggle_code_it_data_df_drug_경구약제조합_5000종_250922_0952.pkl")

In [ ]:
raise ValueError("stop")

In [ ]:

# 저장된 파일에서 DataFrame 읽기
df_train_test = pd.read_pickle(r"D:\dataset\kaggle_code_it_data_경구약제조합_5000종_250922_0952.pkl")
df_drug = pd.read_pickle(r"D:\dataset\kaggle_code_it_data_df_drug_경구약제조합_5000종_250922_0952.pkl")

df_train_test['category_id'] = df_train_test['drug_N'].apply(drugN_to_category_id)
df_drug['category_id'] = df_drug['drug_N'].apply(drugN_to_category_id)

print("df_train_test shape:", df_train_test.shape)
print("df_drug shape:", df_drug.shape)

In [ ]:
df_train_test

In [ ]:
pbar = tqdm(df_train_test.iterrows(), total=len(df_train_test), mininterval=3, desc="Checking image files")
for index, rw in pbar:
    if os.path.exists(rw['imgfile']) == False:
        print(rw)
        df_train_test.drop(rw.index, inplace=True)

In [ ]:
print('df_drug:', len(df_drug))

In [9]:
def create_yolo_dataset(df, yolo_dataset_path, use_jpg=True):
    """
                                              filename  ext                                             file_name                                                                                                                                                                             path                         label    drug0    drug1    drug2    drug3 Train  Test   drug_N width height bbox_x bbox_y bbox_w bbox_h yolo_x yolo_y yolo_w yolo_h class_id   set_type
  872 K-001900-016548-018110-027926_0_2_0_2_75_000_200 json K-001900-016548-018110-027926_0_2_0_2_75_000_200.json d:\dataset\local_kaggle_code_it_data\ai04-level1-project.zip.unzip\train_annotations\K-001900-016548-018110-027926_json\K-001900\K-001900-016548-018110-027926_0_2_0_2_75_000_200.json K-001900-016548-018110-027926 K-001900 K-016548 K-018110 K-027926 False False K-001900   976   1280    142    241    200    127  0.248 0.2379 0.2049 0.0992        1 Validation
  870 K-001900-016548-018110-027926_0_2_0_2_70_000_200 json K-001900-016548-018110-027926_0_2_0_2_70_000_200.json d:\dataset\local_kaggle_code_it_data\ai04-level1-project.zip.unzip\train_annotations\K-001900-016548-018110-027926_json\K-001900\K-001900-016548-018110-027926_0_2_0_2_70_000_200.json K-001900-016548-018110-027926 K-001900 K-016548 K-018110 K-027926 False False K-001900   976   1280    630    894    211    133 0.7536 0.7504 0.2162 0.1039        1 Validation

dataset/
├── images/
│   ├── train/
│   │   ├── image1.jpg
│   │   ├── image2.jpg
│   │   └── ...
│   ├── val/
│   │   ├── val_image1.jpg
│   │   └── ...
│   └── test/ (선택적)
└── labels/
    ├── train/
    │   ├── image1.txt
    │   ├── image2.txt
    │   └── ...
    ├── val/
    │   ├── val_image1.txt
    │   └── ...
    └── test/ (선택적)

    DataFrame df_convert_yolo를 만들고 df를 복사하고 to yolo_image_train, yolo_image_val, yolo_label_train, yolo_label_val를 만든다.
    df의 Train 컬럼을 참고 하면 된다.
    df_convert_yolo를 yolo_dataset_path에 저장한다. (원복등에 참고 할 수 있을 것이다.)

    만들어진 df_convert_yolo 를 이용하여 images를 yolo_image_train, yolo_image_val에 이동시킨다.
    전체적인 데이타 용량이 큼으로 파일을 이동시키는 방식으로 한다.

    """

    images_train_dir = os.path.join(yolo_dataset_path, 'images', 'train')
    images_val_dir = os.path.join(yolo_dataset_path, 'images', 'val')
    images_test_dir = os.path.join(yolo_dataset_path, 'images', 'test')
    labels_train_dir = os.path.join(yolo_dataset_path, 'labels', 'train')
    labels_val_dir = os.path.join(yolo_dataset_path, 'labels', 'val')
    labels_test_dir = os.path.join(yolo_dataset_path, 'labels', 'test')

    df.sort_values('filename', inplace=True)
    df = df.reset_index(drop=True)

    pbar = tqdm(df.iterrows(), total=len(df), mininterval=3, desc="Creating YOLO dataset")
    for idx, row in pbar:
        set_type = row.get('set_type', 'Unknown')
        img_dst, label_dst = None, None

        filename = row['filename']
        label_name = row['label']
        # 파일명에 _순서 붙이기
        if use_jpg:
            base_img_name = f"{filename}.jpg"
        else:
            base_img_name = f"{filename}.png"
        base_label_name = f"{filename}.txt"

        if set_type == 'Train':
            img_dst = os.path.join(images_train_dir, base_img_name)
            label_dst = os.path.join(labels_train_dir, base_label_name)
        elif set_type == 'Validation':
            img_dst = os.path.join(images_val_dir, base_img_name)
            label_dst = os.path.join(labels_val_dir, base_label_name)
        elif set_type == 'Test':
            img_dst = os.path.join(images_test_dir, base_img_name)
            label_dst = os.path.join(labels_test_dir, base_label_name)
        else:
            continue

        df.loc[idx, 'yolo_image'] = img_dst
        df.loc[idx, 'yolo_label'] = label_dst

        if idx % 100 == 0:
            pbar.set_postfix_str(base_img_name)

    return df

df = df_train_test.copy()
df_yolo = create_yolo_dataset(df, r'D:\dataset\yolo_dataset_경구약제조합_5000종_250921_1433', use_jpg=True)

NameError: name 'df_train_test' is not defined

In [ ]:
df_yolo

In [ ]:
# drug_N
df_yolo.head_att(10)

In [10]:
class DFYOLOToClassificationDataset(Dataset):

    def __init__(self, df_yolo):
        self.df_yolo = df_yolo
        self.data = []
        self._load_data()

    def _load_data(self):
        pbar = tqdm(self.df_yolo.iterrows(), total=len(self.df_yolo), mininterval=3, desc="Loading dataset")
        for idx, row in pbar:
            # print('-' * 80)
            # print(row)
            # print('-' * 80)
            
            image_path = row['imgfile']
            category_id  = row['category_id']
            class_id = row['class_id']
            drug_N = row['drug_N']
            width = row['width']
            height = row['height']
            bbox_x = row['bbox_x']
            bbox_y = row['bbox_y']
            bbox_w = row['bbox_w']
            bbox_h = row['bbox_h']
            yolo_x = row['yolo_x']
            yolo_y = row['yolo_y']
            yolo_w = row['yolo_w']
            yolo_h = row['yolo_h']
            
            # 좌표 검증
            try:
                img = Image.open(image_path)
                img_w, img_h = img.size
                x1 = int(bbox_x)
                y1 = int(bbox_y)
                x2 = int(bbox_x + bbox_w)
                y2 = int(bbox_y + bbox_h)
                # 이미지 내에 bbox가 있는지 체크
                if x2 > x1 and y2 > y1 and x1 >= 0 and y1 >= 0 and x2 <= img_w and y2 <= img_h:
                    self.data.append({
                        'image_path': image_path,
                        'class_id': class_id,
                        'category_id': category_id,
                        'drug_N': drug_N,
                        'size': (width, height),
                        'bbox': (bbox_x, bbox_y, bbox_w, bbox_h),
                        'yolo_bbox': (yolo_x, yolo_y, yolo_w, yolo_h)
                    })
                else:
                    # 잘못된 bbox는 추가하지 않음
                    continue
            except Exception as e:
                # 이미지 열기 오류 등은 추가하지 않음
                continue

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        target = self.data[idx]
        image = Image.open(target['image_path']).convert('RGB')
        img_w, img_h = image.size
        bbox_x, bbox_y, bbox_w, bbox_h = target['bbox']
        x1 = int(bbox_x)
        y1 = int(bbox_y)
        x2 = int(bbox_x + bbox_w)
        y2 = int(bbox_y + bbox_h)
        image = image.crop((x1, y1, x2, y2))
        return image, target
    
    def gettarget(self, idx):
        target = self.data[idx]
        return target
    
    
dataset = DFYOLOToClassificationDataset(df_yolo)
print(f"Dataset size: {len(dataset)}")

NameError: name 'df_yolo' is not defined

In [ ]:
raise ValueError("stop")

In [11]:
from collections import defaultdict
def sample_by_category(dataset, n=10):
    category_samples = defaultdict(list)
    pbar = tqdm(range(len(dataset)), total=len(dataset), mininterval=3, desc="Sampling by category")
    for i in pbar:
        target = dataset.gettarget(i)
        cid = target['category_id']
        if len(category_samples[cid]) < n:
            category_samples[cid].append((i, target))
    # flatten
    samples = []
    for v in category_samples.values():
        samples.extend(v)
    return samples
dataset_sample_category = sample_by_category(dataset, n=100)
print(f"샘플 개수: {len(dataset_sample_category)}")

NameError: name 'dataset' is not defined

In [ ]:
image, target = dataset_sample_category[0]
print(target)

In [ ]:
from collections import defaultdict
import cv2
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances

def sample_by_category_with_filtering(dataset, n=10, outlier_threshold=2.0):
    """
    카테고리별로 n개씩 샘플링하고 히스토그램 기반으로 이상치 제거
    """
    category_samples = defaultdict(list)
    
    # 1단계: 카테고리별로 샘플 수집
    print("1단계: 카테고리별 샘플 수집")
    pbar = tqdm(range(len(dataset)), total=len(dataset), mininterval=3, desc="Sampling by category")
    for i in pbar:
        target = dataset.gettarget(i)
        cid = target['category_id']
        if len(category_samples[cid]) < n:
            category_samples[cid].append((i, target))
    
    print(f"카테고리 수: {len(category_samples)}")
    
    # 2단계: 각 카테고리별로 히스토그램 필터링
    filtered_samples = []
    category_stats = {}
    
    print("2단계: 히스토그램 기반 필터링")
    for cid, samples in tqdm(category_samples.items(), desc="Filtering by histogram"):
        if len(samples) < 2:  # 샘플이 너무 적으면 필터링 안함
            filtered_samples.extend(samples)
            category_stats[cid] = {'original': len(samples), 'filtered': len(samples), 'removed': 0}
            continue
        
        # 히스토그램 계산
        histograms = []
        valid_samples = []
        
        for idx, target in samples:
            try:
                image, _ = dataset[idx]
                img_array = np.array(image)
                
                # RGB 히스토그램 계산 (간단하게 각 채널 32bins)
                hist_r = cv2.calcHist([img_array], [0], None, [32], [0, 256])
                hist_g = cv2.calcHist([img_array], [1], None, [32], [0, 256])
                hist_b = cv2.calcHist([img_array], [2], None, [32], [0, 256])
                
                # 정규화된 히스토그램 결합
                hist_combined = np.concatenate([hist_r.flatten(), hist_g.flatten(), hist_b.flatten()])
                hist_normalized = hist_combined / (hist_combined.sum() + 1e-8)
                
                histograms.append(hist_normalized)
                valid_samples.append((idx, target))
                
            except Exception as e:
                continue
        
        if len(histograms) < 2:
            filtered_samples.extend(valid_samples)
            category_stats[cid] = {'original': len(samples), 'filtered': len(valid_samples), 'removed': 0}
            continue
        
        # 히스토그램 간 평균 거리 계산
        hist_matrix = np.array(histograms)
        distances = euclidean_distances(hist_matrix, hist_matrix)
        mean_distances = np.mean(distances, axis=1)
        
        # 평균 + 표준편차 * threshold를 이상치 기준으로 사용
        mean_dist = np.mean(mean_distances)
        std_dist = np.std(mean_distances)
        threshold_value = mean_dist + outlier_threshold * std_dist
        
        # 필터링
        category_filtered = []
        for i, (sample, dist) in enumerate(zip(valid_samples, mean_distances)):
            if dist <= threshold_value:
                category_filtered.append(sample)
        
        filtered_samples.extend(category_filtered)
        category_stats[cid] = {
            'original': len(samples),
            'filtered': len(category_filtered),
            'removed': len(valid_samples) - len(category_filtered)
        }
    
    # 결과 출력
    total_original = sum(stats['original'] for stats in category_stats.values())
    total_filtered = sum(stats['filtered'] for stats in category_stats.values())
    total_removed = sum(stats['removed'] for stats in category_stats.values())
    
    print(f"\n=== 필터링 결과 ===")
    print(f"원본 샘플: {total_original}개")
    print(f"필터링 후: {total_filtered}개")
    print(f"제거된 이상치: {total_removed}개")
    
    # 카테고리별 상세 통계 (제거가 많이 된 상위 5개)
    high_removal_categories = sorted(category_stats.items(), 
                                   key=lambda x: x[1]['removed'], reverse=True)[:5]
    
    if high_removal_categories and high_removal_categories[0][1]['removed'] > 0:
        print("\n제거가 많이 된 카테고리 (상위 5개):")
        for cid, stats in high_removal_categories:
            if stats['removed'] > 0:
                print(f"  카테고리 {cid}: {stats['original']} → {stats['filtered']} ({stats['removed']}개 제거)")
    
    return filtered_samples

# 실행
dataset_sample_category_filtered = sample_by_category_with_filtering(
    dataset, 
    n=100,
    outlier_threshold=1.0  # 값을 낮추면 더 엄격하게 필터링
)

print(f"\n최종 샘플 개수: {len(dataset_sample_category_filtered)}")

In [ ]:
import os
import json
from datetime import datetime

def save_sample_category_to_coco(dataset_sample_category, output_dir, image_format='jpg'):
    """
    dataset_sample_category를 COCO 형식으로 저장하는 함수
    Args:
        dataset_sample_category: [(idx, target)] 리스트
        output_dir: 저장 폴더 경로
        image_format: 저장 이미지 확장자
    """
    os.makedirs(output_dir, exist_ok=True)
    images_dir = os.path.join(output_dir, 'images')
    os.makedirs(images_dir, exist_ok=True)

    coco_data = {
        "info": {
            "description": "Sample category dataset",
            "version": "1.0",
            "year": datetime.now().year,
            "contributor": "SampleCategory",
            "date_created": datetime.now().isoformat()
        },
        "licenses": [
            {"id": 1, "name": "Unknown", "url": ""}
        ],
        "images": [],
        "annotations": [],
        "categories": []
    }

    # category_id 목록 추출
    category_id_to_drugN = {}
    for _, target in dataset_sample_category:
        cid = target['category_id']
        if cid not in category_id_to_drugN:
            category_id_to_drugN[cid] = target['drug_N']    
    category_ids = sorted({target['category_id'] for _, target in dataset_sample_category})
    for cid in category_ids:
        coco_data["categories"].append({
            "id": cid,
            "name": category_id_to_drugN[cid],
            "supercategory": "pill"
        })

    annotation_id = 1
    pbar = tqdm(dataset_sample_category, total=len(dataset_sample_category), mininterval=3, desc="Saving images and annotations")
    for idx, target in pbar:
        # 이미지 crop
        image = Image.open(target['image_path']).convert('RGB')
        bbox_x, bbox_y, bbox_w, bbox_h = target['bbox']
        x1, y1 = int(bbox_x), int(bbox_y)
        x2, y2 = int(bbox_x + bbox_w), int(bbox_y + bbox_h)
        crop_img = image.crop((x1, y1, x2, y2))

        # 파일명 생성 및 저장
        base_name = os.path.splitext(os.path.basename(target['image_path']))[0]
        #image_filename = f"{target['category_id']:03d}_{idx:06d}_{base_name}.{image_format}"
        image_filename = f"{target['category_id']:03d}_{idx:06d}.{image_format}"
        image_path = os.path.join(images_dir, image_filename)
        crop_img.save(image_path)
        crop_w, crop_h = crop_img.size

        # COCO 이미지 정보
        coco_data["images"].append({
            "id": idx + 1,
            "width": crop_w,
            "height": crop_h,
            "file_name": image_filename,
            "license": 1,
            "date_captured": datetime.now().isoformat()
        })

        # COCO annotation 정보
        coco_data["annotations"].append({
            "id": annotation_id,
            "image_id": idx + 1,
            "category_id": target['category_id'],
            "segmentation": [],
            "area": crop_w * crop_h,
            "bbox": [0, 0, crop_w, crop_h],
            "iscrowd": 0
        })
        annotation_id += 1

    # JSON 저장
    json_path = os.path.join(output_dir, "instances_sample_category.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(coco_data, f, indent=2, ensure_ascii=False)

    print(f"COCO 저장 완료: {json_path}, 이미지 {len(coco_data['images'])}개")

# 사용 예시
sample_category_coco_path = r"D:\dataset\kaggle_code_it_data_category_coco_his"
if os.path.exists(sample_category_coco_path):
    shutil.rmtree(sample_category_coco_path)
save_sample_category_to_coco(dataset_sample_category, sample_category_coco_path)

In [12]:
def clean_coco_json(json_path, images_dir, output_json_path=None):
    """
    COCO JSON 파일에서 실제 존재하지 않는 이미지에 대한 annotation을 제거하고 정리
    
    Args:
        json_path: COCO JSON 파일 경로
        images_dir: 이미지 폴더 경로
        output_json_path: 출력 JSON 파일 경로 (None이면 원본 파일에 덮어쓰기)
    """
    import os
    import json
    from datetime import datetime
    
    # JSON 파일 로드
    with open(json_path, 'r', encoding='utf-8') as f:
        coco_data = json.load(f)
    
    print(f"원본 데이터: 이미지 {len(coco_data['images'])}개, annotation {len(coco_data['annotations'])}개")
    
    # 실제 존재하는 이미지 파일들 확인
    existing_images = []
    removed_images = []
    
    for img_info in tqdm(coco_data['images'], desc="Checking image files"):
        image_path = os.path.join(images_dir, img_info['file_name'])
        if os.path.exists(image_path):
            existing_images.append(img_info)
        else:
            removed_images.append(img_info)
            print(f"Missing image: {img_info['file_name']}")
    
    # 존재하는 이미지의 ID 목록
    existing_image_ids = {img['id'] for img in existing_images}
    
    # 존재하는 이미지에 대한 annotation만 유지
    existing_annotations = []
    removed_annotations = []
    
    for ann in coco_data['annotations']:
        if ann['image_id'] in existing_image_ids:
            existing_annotations.append(ann)
        else:
            removed_annotations.append(ann)
    
    # 사용된 카테고리 ID만 유지
    used_category_ids = {ann['category_id'] for ann in existing_annotations}
    existing_categories = [cat for cat in coco_data['categories'] if cat['id'] in used_category_ids]
    
    # 정리된 COCO 데이터 생성
    cleaned_coco_data = {
        "info": {
            **coco_data.get("info", {}),
            "description": f"{coco_data.get('info', {}).get('description', '')} (cleaned)",
            "date_created": datetime.now().isoformat()
        },
        "licenses": coco_data.get("licenses", []),
        "images": existing_images,
        "annotations": existing_annotations,
        "categories": existing_categories
    }
    
    # 결과 출력
    print(f"\n=== 정리 결과 ===")
    print(f"이미지: {len(coco_data['images'])} → {len(existing_images)} ({len(removed_images)}개 제거)")
    print(f"annotation: {len(coco_data['annotations'])} → {len(existing_annotations)} ({len(removed_annotations)}개 제거)")
    print(f"카테고리: {len(coco_data['categories'])} → {len(existing_categories)}")
    
    if removed_images:
        print(f"\n제거된 이미지 목록 (처음 10개):")
        for img in removed_images[:10]:
            print(f"  - {img['file_name']}")
    
    # JSON 저장
    if output_json_path is None:
        output_json_path = json_path
    
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(cleaned_coco_data, f, indent=2, ensure_ascii=False)
    
    print(f"\n정리된 JSON 저장: {output_json_path}")
    
    return cleaned_coco_data


sample_category_coco_path = r"D:\dataset\kaggle_code_it_data_category_coco_100_all_clean_250922"
json_path = os.path.join(sample_category_coco_path, "instances_category.json")
images_dir = os.path.join(sample_category_coco_path, "images")

# JSON 정리 실행
cleaned_coco_data = clean_coco_json(json_path, images_dir)

원본 데이터: 이미지 11706개, annotation 11706개


Checking image files:  24%|██▍       | 2812/11706 [00:00<00:00, 27939.77it/s]

Missing image: 6191_001479.jpg
Missing image: 6191_001495.jpg
Missing image: 12777_000075.jpg
Missing image: 12777_000082.jpg
Missing image: 12777_000781.jpg
Missing image: 19551_000852.jpg
Missing image: 19551_000978.jpg
Missing image: 23222_000239.jpg
Missing image: 5001_001001.jpg
Missing image: 5001_001052.jpg
Missing image: 4377_000830.jpg
Missing image: 5885_000447.jpg
Missing image: 13394_001498.jpg
Missing image: 22361_000632.jpg
Missing image: 22361_001487.jpg
Missing image: 1865_002276.jpg
Missing image: 3742_002896.jpg
Missing image: 25437_002671.jpg
Missing image: 1899_005394.jpg
Missing image: 1899_005395.jpg
Missing image: 1899_005466.jpg
Missing image: 1899_005522.jpg
Missing image: 1899_005666.jpg
Missing image: 1899_005705.jpg
Missing image: 16547_005388.jpg
Missing image: 16547_005392.jpg
Missing image: 16547_005440.jpg
Missing image: 16547_005463.jpg
Missing image: 16547_005560.jpg
Missing image: 4542_005626.jpg
Missing image: 3543_005393.jpg
Missing image: 3543_0053

Checking image files:  73%|███████▎  | 8591/11706 [00:00<00:00, 28124.34it/s]

Missing image: 32309_025168.jpg
Missing image: 31884_012316.jpg
Missing image: 31884_012319.jpg
Missing image: 31884_012321.jpg
Missing image: 31884_012322.jpg
Missing image: 27776_012196.jpg
Missing image: 23202_016708.jpg
Missing image: 23202_016710.jpg
Missing image: 23202_022606.jpg
Missing image: 4999_023567.jpg
Missing image: 20851_021395.jpg
Missing image: 24751_021361.jpg
Missing image: 23318_028656.jpg
Missing image: 6834_028924.jpg
Missing image: 6834_028928.jpg
Missing image: 6834_028951.jpg
Missing image: 25199_028325.jpg
Missing image: 29710_027423.jpg
Missing image: 29710_027425.jpg
Missing image: 29710_028457.jpg
Missing image: 30849_029044.jpg
Missing image: 30849_029106.jpg
Missing image: 30849_029159.jpg
Missing image: 38926_028519.jpg
Missing image: 41148_027992.jpg
Missing image: 41148_027997.jpg
Missing image: 41148_028001.jpg
Missing image: 41148_028053.jpg
Missing image: 41148_028112.jpg
Missing image: 41148_028706.jpg
Missing image: 41148_028710.jpg
Missing imag

Checking image files: 100%|██████████| 11706/11706 [00:00<00:00, 28059.84it/s]


Missing image: 38722_047753.jpg

=== 정리 결과 ===
이미지: 11706 → 11587 (119개 제거)
annotation: 11706 → 11587 (119개 제거)
카테고리: 118 → 118

제거된 이미지 목록 (처음 10개):
  - 6191_001479.jpg
  - 6191_001495.jpg
  - 12777_000075.jpg
  - 12777_000082.jpg
  - 12777_000781.jpg
  - 19551_000852.jpg
  - 19551_000978.jpg
  - 23222_000239.jpg
  - 5001_001001.jpg
  - 5001_001052.jpg

정리된 JSON 저장: D:\dataset\kaggle_code_it_data_category_coco_100_all_clean_250922\instances_category.json


In [ ]:

sample_category_coco_path = r"D:\dataset\kaggle_code_it_data_category_coco_100_all_clean_250922"
json_path = os.path.join(sample_category_coco_path, "instances_category.json")
images_dir = os.path.join(sample_category_coco_path, "images")

# JSON 정리 실행
cleaned_coco_data = clean_coco_json(json_path, images_dir)
